# nb_00_setup
Creates everything the pipeline needs in `lh_retail`. **Idempotent**: safe to run any time.

1. Schemas `bronze`, `silver`, `gold`, `ops`
2. Ops tables (`run_log`, `watermark`, `file_log`, `dq_results`)
3. Folders `Files/landing`, `Files/_state`

Parameter `reset = True` wipes all data (**dev only**) so you can start the project from scratch.

In [ ]:
reset = False
run_date = ""

In [ ]:
%run nb_config

In [ ]:
%run nb_lib_logging

In [ ]:
%run nb_lib_delta

In [ ]:
with RunLogger("nb_00_setup", run_date) as run:
    log = run.log

    # --- Optional reset (dev only) -----------------------------------------
    if reset:
        if ENV == "prod":
            raise RuntimeError("reset=True is not allowed in prod")
        log.warning("RESET requested: dropping all schemas and landing files in %s", ENV)
        for schema in SCHEMAS:
            spark.sql(f"DROP SCHEMA IF EXISTS {schema} CASCADE")
        for folder in (LANDING_ROOT, STATE_ROOT):
            if notebookutils.fs.exists(folder):
                notebookutils.fs.rm(folder, True)

    # --- 1. Schemas --------------------------------------------------------
    for schema in SCHEMAS:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    log.info("schemas ready: %s", SCHEMAS)

    # --- 2. Ops tables -----------------------------------------------------
    for name, ddl in OPS_TABLES.items():
        spark.sql(f"CREATE TABLE IF NOT EXISTS {tbl('ops', name)} ({ddl}) USING DELTA")
    log.info("ops tables ready: %s", list(OPS_TABLES))

    # --- 3. Folders --------------------------------------------------------
    for folder in (LANDING_ROOT, STATE_ROOT):
        notebookutils.fs.mkdirs(folder)
    log.info("folders ready: %s, %s", LANDING_ROOT, STATE_ROOT)

    run.metric("schemas", len(SCHEMAS))
    run.metric("ops_tables", len(OPS_TABLES))

## Verify

In [ ]:
for schema in SCHEMAS:
    tables = [t.name for t in spark.catalog.listTables(schema)]
    print(f"{schema:7s}: {tables}")

display(spark.sql("SELECT job_name, status, start_ts, duration_sec, metrics_json "
                  "FROM ops.run_log ORDER BY start_ts DESC LIMIT 5"))

In [ ]:
notebookutils.notebook.exit(run.exit_value())